# **AI Job Agent — PHASE 9: Evaluation**
**Goal:** Measure whether the Phase 8 agent actually behaves the way its system prompt promises -- picks the right tool for the right request, never invents a job or skill that isn't in the Phase 7 data, never presents 'match_score' as a probability, and fails gracefully rather than hallucinating when asked about something it doesn't have.

**What checks automatically vs by hand:**
- **Automated:** which tool was called for each prompt, whether any job_id mentioned actually exists in the loaded data, whether forbidden phrasing (percentages, guarantees) shows up in reply.
- **Manual:** whether the content of a grounded reply is actually well-written and helpful.

**Phase 9 output:** `evaluation_report.json` + `evaluation_results.csv`, for the final report.

# 1. **Setup**

In [22]:
!pip install -q langgraph langchain-openai langchain-core pydantic openai

import os
import json
import uuid
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Literal

import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from google.colab import userdata


OPENAI_API_KEY = userdata.get("OPENAI_API_KEY").strip()
assert OPENAI_API_KEY, "OPENAI_API_KEY was not found in Colab Secrets."
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

AGENT_MODEL = "gpt-5.6-luna"
_client = OpenAI()

print("Setup completed successfully.")

Setup completed successfully.


## 2. Data Contracts(Schemas)

identical to Phase 8 section 2

In [23]:
ExperienceLevel = Literal[
    "Internship", "Entry Level", "Mid Level", "Senior Level",
    "Executive", "Not Specified",
]

EducationLevel = Literal[
    "High School", "Diploma", "Bachelor's Degree",
    "Master's Degree", "PhD", "Not Specified",
]

EmploymentType = Literal[
    "Full-time", "Part-time", "Contract", "Internship",
    "Temporary", "Not Specified",
]

WorkArrangement = Literal["On-site", "Remote", "Hybrid", "Not Specified"]

print("Enums defined successfully.")


Enums defined successfully.


In [24]:
class CandidatePreferences(BaseModel):
  desired_countries: List[str] = Field(default_factory=list)
  desired_cities: List[str] = Field(default_factory=list)
  work_arrangement_preference: WorkArrangement = "Not Specified"
  employment_type_preference: EmploymentType = "Full-time"
  willing_to_relocate: bool = False
  nationality: Optional[str] = None
  salary_expectation_min: Optional [float] = None
  salary_expectation_max: Optional [float] = None
  salary_currency: Optional[str] = None
  salary_period: Literal["hour", "day", "month", "year", "Not Specified"] = "Not Specified"
  availability:  Optional[str] = None
  notes: str = ""



class CandidateProfile(BaseModel):
  candidate_id: str
  most_recent_title: Optional[str] = None
  past_titles: List[str] = Field(default_factory=list)

  hard_skills_norm: List[str] = Field(default_factory=list)
  soft_skills_norm: List[str] = Field(default_factory=list)
  tools_technologies_norm: List[str] = Field(default_factory=list)
  languages_norm: List[str] = Field(default_factory=list)
  all_skills_norm: List[str] = Field(default_factory=list)

  qualifications: List[str] = Field(default_factory=list)
  experience_level: ExperienceLevel = "Not Specified"
  years_experience_total: Optional [float] = None
  education_level: List[EducationLevel] = Field(default_factory=list)
  education_field: List[str] = Field(default_factory=list)
  industries: List[str] = Field(default_factory=list)

  extraction_confidence: float = 1.0

  preferences: CandidatePreferences = Field(default_factory=CandidatePreferences)


print("Candidate profile schema created successfully.")


Candidate profile schema created successfully.


In [25]:
class JobLocation(BaseModel):
  city: Optional[str] = None
  country: Optional[str] = None

class MatchedJob(BaseModel):
  job_id: str
  phase6_rank: int
  phase5_rank: int
  title: str
  company: Optional[str] = None
  url: Optional[str] = None
  location:  JobLocation =  Field(default_factory=JobLocation)
  job_description: str

  match_score: Optional [float] = None
  objective_rank_score: Optional [float] = None
  scoring_confidence: Optional [float] = None

  reranker_raw_score: Optional [float] = None
  reranker_percentile: Optional [float] = None

  matched_core_skills: List[str] = Field(default_factory=list)
  missing_core_skills: List[str] = Field(default_factory=list)
  matched_qualifications: List[str] = Field(default_factory=list)
  missing_qualifications: List[str] = Field(default_factory=list)
  matched_education_fields: List[str] = Field(default_factory=list)
  missing_education_fields: List[str] = Field(default_factory=list)


class FinalTopJobs(BaseModel):
  candidate_id: str
  created_utc: str
  jobs: List[MatchedJob] = Field(default_factory=list)

print("Job matching schema created successfully.")


Job matching schema created successfully.


In [26]:
class LearningArea(BaseModel):
  missing_skill: str
  learning_area: str

class Phase7_JopOutput(BaseModel):
  job_id: str
  title: str
  company: Optional[str] = None
  url: Optional[str] = None
  location:  JobLocation =  Field(default_factory=JobLocation)

  match_score: float
  scoring_confidence: float

  why_match: str
  matched_skills: List[str] = Field(default_factory=list)
  missing_skills: List[str] = Field(default_factory=list)
  matched_qualifications: List[str] = Field(default_factory=list)
  missing_qualifications: List[str] = Field(default_factory=list)
  matched_education: List[str] = Field(default_factory=list)
  missing_education: List[str] = Field(default_factory=list)
  learning_areas: List[LearningArea] = Field(default_factory=list)


class Phase7_Output(BaseModel):
  candidate_id: str
  created_utc: str
  pjase: str= "Phase 7"
  jobs: List[Phase7_JopOutput] = Field(default_factory=list)

  def get(self, job_id: str) -> Optional["Phase7_JopOutput"] :
    return next((j for j in self.jobs if j.job_id == job_id), None)

  print("Phase 7 explanation schema created successfuly")

Phase 7 explanation schema created successfuly


In [27]:
class ImprovementAction(BaseModel):
    action: str
    reason: str
    priority: Literal["high", "medium", "low"]
    category: Literal["skill", "qualification", "resume_edit"]

class ImprovementPlan(BaseModel):
    candidate_id: str
    actions: List[ImprovementAction] = Field(default_factory=list)

class InterviewQuestion(BaseModel):
    question: str
    what_its_testing: str
    suggested_angle: str

class Interviewprep(BaseModel):
    job_id: str
    questions: List[InterviewQuestion] = Field(default_factory=list)

print("Phase 8 output schemas created successfully.")


Phase 8 output schemas created successfully.


## 3. Load The Outputs from Phases 2, 6 or 7.

identical to Phase 8 section 3


In [28]:
from google.colab import files as _colab_files

print("Upload candidate_profile.json from Phase 2")
uploaded_p2 = _colab_files.upload()

P2_DIR = Path("/content/phase2")
P2_DIR.mkdir(exist_ok=True)
for name, content in uploaded_p2.items():
    (P2_DIR / name).write_bytes(content)

p2_file = P2_DIR / "candidate_profile.json"
assert p2_file.exists(), "candidate_profile.json not found among uploaded files."

CANDIDATE_PROFILE = CandidateProfile.model_validate(json.loads(p2_file.read_text(encoding="utf-8")))
print(f"\nLoaded candidate {CANDIDATE_PROFILE.candidate_id} ({CANDIDATE_PROFILE.most_recent_title}).")


Upload candidate_profile.json from Phase 2


Saving candidate_profile.json to candidate_profile (1).json

Loaded candidate e07ca2443daccbb5 (Freelance AI Engineer).


In [29]:

print("Upload final_top_jobs.json from Phase 6")
uploaded_p6 = _colab_files.upload()

P6_DIR = Path("/content/phase6")
P6_DIR.mkdir(exist_ok=True)
for name, content in uploaded_p6.items():
    (P6_DIR / name).write_bytes(content)

p6_file = P6_DIR / "final_top_jobs.json"
assert p6_file.exists(), "final_top_jobs.json not found among uploaded files."

FINAL_TOP_JOBS = FinalTopJobs.model_validate(json.loads(p6_file.read_text(encoding="utf-8")))
print(f"\nLoaded {len(FINAL_TOP_JOBS.jobs)} matched jobs for candidate {FINAL_TOP_JOBS.candidate_id}.")

Upload final_top_jobs.json from Phase 6


Saving final_top_jobs.json to final_top_jobs (1).json

Loaded 1 matched jobs for candidate e07ca2443daccbb5.


In [30]:

print("Upload the Phase7_output.json from phase 7:")
uploaded_p7 = _colab_files.upload()

P7_DIR = Path("/content/phase7")
P7_DIR.mkdir(exist_ok=True)
for name, content in uploaded_p7.items():
    (P7_DIR / name).write_bytes(content)

p7_file = P7_DIR / "phase7_output.json"
assert p7_file.exists(), "phase7_output.json not found among uploaded files."

PHASE7_OUTPUT = Phase7_Output.model_validate(json.loads(p7_file.read_text(encoding="utf-8")))
print(f"\nLoaded {len(PHASE7_OUTPUT.jobs)} real Phase 7 explanations.")

Upload the Phase7_output.json from phase 7:


Saving phase7_output.json to phase7_output (1).json

Loaded 1 real Phase 7 explanations.


## 4. Rebuild the Phase 8 agent

In [36]:
def generate_improvement_plan(candidate_profile: CandidateProfile, explanation: Phase7_Output) -> ImprovementPlan:
    """Generate an improvement plan based on the candidate profile and Phase 7 output."""
    skill_gap_counts: dict = {}
    qual_gap_counts: dict = {}
    edu_gap_counts: dict = {}

    for job in explanation.jobs:
        for la in job.learning_areas:
            skill_gap_counts[la.learning_area] = skill_gap_counts.get(la.learning_area, 0) + 1
        for q in job.missing_qualifications:
            qual_gap_counts[q] = qual_gap_counts.get(q, 0) + 1
        for edu in job.missing_education:
            edu_gap_counts[edu] = edu_gap_counts.get(edu, 0) + 1

        actions: List[ImprovementAction] = []
        n_jobs = max(len(explanation.jobs), 1)

        def _priority(count: int) -> str:
            freq = count / n_jobs
            if freq >= 0.5:
                return "high"
            elif freq >= 0.2:
                return "medium"
            else:
                return "low"


        for area, count in sorted(skill_gap_counts.items(), key=lambda kv: -kv[1]):
            actions.append(ImprovementAction(
                action=f"Learn or strengthen: {area}",
                reason=f"Missing from {count}/{n_jobs} of your top-matched roles.",
                priority=_priority(count),
                category="skill"
                ))

        for qual, count in sorted(qual_gap_counts.items(), key=lambda kv: -kv[1]):
            actions.append(ImprovementAction(
                action=f"Consider obtaining: {qual}",
                reason=f"Requested by {count}/{n_jobs} of your top-matched roles.",
                priority=_priority(count),
                category="qualification"
                ))

        for field_name, count in sorted(edu_gap_counts.items(), key=lambda kv: -kv[1]):
            actions.append(ImprovementAction(
                action=f"Education gap noted: {field_name}",
                reason=f"Requested by {count}/{n_jobs} of your top-matched roles."
                       f"Consider whether a course or certificate can partially close this gap.",
                priority=_priority(count),
                category="qualification"
                ))

        if candidate_profile.extraction_confidence < 0.6:
            actions.append(ImprovementAction(
                action="Review and tighten your resume's structure and section headers.",
                reason="Low extraction confidence suggests the resume may be hard to parse or thin on details.",
                priority="high",
                category="resume_edit"
            ))

        return ImprovementPlan(candidate_id=candidate_profile.candidate_id, actions=actions)
print("generated_Improvement_plan defined successfully.")

generated_Improvement_plan defined successfully.


In [37]:

def generate_interview_questions(job: Phase7_Output,candidate_profile: CandidateProfile ) -> Interviewprep:
    """Generate interview questions based on the Phase 7 output."""

    system = (
        "You write interview-preparation questions for a job candidate. "
        "Base every question ONLY on the job title, why_match narrative, and "
        "the candidate's listed matched/missing skills given below. Never"
        "invent facts about the company."
        "Return strict JSON matching: "
        '{"questions":[{"question": str, "what_its_testing": str, "suggested_angle": str}]}'
    )

    user = json.dumps({
        "job_title": job.title,
        "why_match": job.why_match,
        "candidate_matched_skills": job.matched_skills,
        "candidate_missing_skills": job.missing_skills,
    })

    response = _client.chat.completions.create(
        model= AGENT_MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        response_format={"type": "json_object"},)

    data = json.loads(response.choices[0].message.content)
    questions = [InterviewQuestion(**q) for q in data["questions"]]
    return Interviewprep(job_id=job.job_id, questions=questions)

print("generate_interview_questions defined successfully.")



generate_interview_questions defined successfully.


In [38]:
SYSTEM_PROMPT = """
You are the conversational agent in an AI Job Matching system, talking
directly to the candidate (candidate_id: {candidate_id}).

You have access to tools that query already-computed matching results from
Phase 5 (scoring), Phase 6 (re-ranking), and Phase 7 (LLM explanations).
You do NOT compute or estimate match scores yourself — you only report what
the tools return.

STRICT GROUNDING RULES

1. Never state a match score, skill, qualification, or education fact about
   a specific job without first calling a tool to retrieve it.
2. Never invent a job_id. Only reference job_ids returned by a tool.
3. If a tool returns an error (e.g. job_not_found, no_browse_pool), explain
   the limitation to the user in plain language — do not pretend the data
   exists.
4. If the user asks something outside your tools' scope (e.g. "change my
   desired salary"), explain that this requires re-running earlier phases
   of the pipeline, not something you can do here.
5. Refer to jobs by title and company (and city if helpful). Do NOT show
   job_ids to the user unless they explicitly ask for them. Keep the
   job_ids in mind internally so you can call tools for follow-up
   questions about a specific job.
6. Keep answers concise and in the user's language (Arabic or English,
   mirror what they use).
7. You may call more than one tool in sequence if a question needs it.
8. Do not repeat match scores when explaining or comparing jobs because match
   scores are already displayed separately in the user interface. Never
   describe a match score as a probability, chance, likelihood of getting the
   job, or likelihood of being hired.
9. Speak naturally to the user. Never use internal wording such as
   "evidence", "tool", "Phase 5", or "the matching evidence". Say things
   like "based on your resume and this job's requirements" instead.
10. Do not infer that the candidate is currently studying, currently enrolled,
    or pursuing a degree unless this is explicitly stated in the tool output.
    When a job requires current study or enrollment, describe it only as a
    job requirement and do not imply that the candidate is currently studying.
11. When explaining a specific job, use the job explanation returned by the
    tool as the source for matched and missing skills, qualifications, and
    education. Do not combine or reinterpret these categories.
12. Do not make your own overall judgment about which job is better, more
    suitable, or a closer match based on comparing tool results. Report the
    differences between jobs factually, such as matched skills, missing
    skills, and match scores when relevant to the user's question. Let the
    candidate make the final judgment.

Be warm and helpful, like a career advisor — but every factual claim must
trace back to a tool call.
    """

print("System prompt defined successfully.")

System prompt defined successfully.


In [50]:
@dataclass

class Session:
    """Holds the outputs across turns for one candidate conversation."""
    preferences: CandidatePreferences = field(default_factory=CandidatePreferences)
    profile: Optional[CandidateProfile] = None
    top_jobs: Optional[FinalTopJobs] = None
    job_explanations: Optional[Phase7_Output] = None

def build_tools(session: Session):
    """Tools closed over 'session' so the agent reads shared state instead
    of passing large objects through the LLM's context every turn."""

    @tool
    def _job_not_found(job_id):
      """Handles cases where a requested job ID can not be found in the
      loaded job results."""
      return {"error": "job_not_found", "job_id": job_id,
                "message": f"No job found with id {job_id} in the loaded results."}

    @tool
    def list_top_jobs_tool(top_n: int=5) -> str:
        """List the candidate's top matched jobs (job_id, title, company,
        location, match_score) from the Phase 7 data already loaded."""
        if session.job_explanations is None:
            return "Error: no job explanations loaded for this session."
        lines = [
            f"{j.job_id}: {j.title} at {j.company} "
            f"({j.location.city}, {j.location.country}) "
            f"- match_score={j.match_score}"
            for j in session.job_explanations.jobs[:top_n]
        ]
        return f"Top {len(lines)} of {len(session.job_explanations.jobs)} jobs:\n" + "\n".join(lines)



    @tool
    def explain_job_tool(job_id: str) -> str:
        """Explain why a specific job is a good fit and what the candidate
         needs, using the Phase 7 explanation already loaded"""
        if session.job_explanations is None:
            return "Error: no job explanations loaded for this session."
        job = session.job_explanations.get(job_id)
        if job is None:
            return f"Error: job_id {job_id} not found in the loaded results."
        needs = f"skills: \n {job.missing_skills} qualifications: \n {job.missing_qualifications} education: \n {job.missing_education}"
        return (
            f"why it fits: {job.why_match}\n"
            f"Match score: {job.match_score} (requirement coverage, not a probability)\n"
            f"What you need: {', '.join(needs) or 'Nothing major missing.'}"
        )

    @tool
    def generate_improvement_plan_tool() -> str:
        """Produce a prioritized list of resume edits, skills to learn, and
        certifications to pursue, based on Phase 7's learning areas and
        gaps aggregated across ALL top-matched jobs."""

        if session.job_explanations is None or session.profile is None:
            return "Error: run search_jobs_tool first."
        plan =  generate_improvement_plan(session.profile, session.job_explanations)
        lines = [f"[{a.priority.upper()}] {a.action} -- {a.reason}" for a in plan.actions]
        return "\n".join(lines) if lines else "No significant gaps found across your top matches."

    @tool
    def generate_interview_questions_tool(job_id: str) -> str:
        """Generate practice interview questions for a specific job the
        candidate has selscted, grounded in that job's Phase 7 explanation."""

        if session.job_explanations is None:
            return "Error: no search has been run yet."
        job = session.job_explanations.get(job_id)
        if job is None:
            return f"Error: job_id {job_id} not found in current results."
        prep = generate_interview_questions(job, session.profile)
        lines = [f"Q: {q.question}\n (Tests: {q.what_its_testing})" for q in prep.questions]
        return "\n".join(lines)

    return[
        _job_not_found,
        list_top_jobs_tool,
        explain_job_tool,
        generate_interview_questions_tool,
        generate_improvement_plan_tool,
    ]

print("Agent tools defined successfully.")

Agent tools defined successfully.


In [51]:
def build_agent(session: Session):
    llm = ChatOpenAI(model=AGENT_MODEL, temperature = 0, reasoning_effort = "none")
    tools = build_tools(session)
    checkpointer = MemorySaver()
    return create_react_agent(
        llm, tools, prompt=SYSTEM_PROMPT, checkpointer=checkpointer,
    )
print("Agent builder defined successfully.")

Agent builder defined successfully.


## 5. Automated grounding checks
Three checks that don't require a human to read every reply:
1. **Tool selection** -> did the agent call the tool the system prompt says it should, for this kind of request?
2. **job_id grounding** -> does every job-id-shaped token in the reply actually exist in the loaded Phase 7 data?. It catche invented jobs.
3. **Forbidden phrasing** -> does the reply state a bare percentage of getting hired, which rule 2 explicitly forbids?

In [52]:
FORBIDDEN_PATTERNS = [
    r"\b\d{1,3}\s?%\s?(chance|probability|likelihood|likely)",
    r"\b(i|we) guarantee\b",
    r"\b(is|are|will be) guaranteed\b",
    r"\byou will (definitely|certainly) get\b",
]

def run_scripted_case(prompt: str, expected_tool: str = None, session: Session = None):
  """Runs one prompt through a fresh agent+session, and returns
  the full result dict for inspection."""

  if session is None:
    session = Session()
    session.profile = CANDIDATE_PROFILE
    session.top_jobs = FINAL_TOP_JOBS
    session.job_explanations = PHASE7_OUTPUT

  app = build_agent(session)
  thread = {"configurable": {"thread_id": str(uuid.uuid4())}}
  result = app.invoke({"messages": [{"role": "user", "content": prompt}]}, config=thread)
  return result, session


def extract_tool_calls(result) -> list:
  """Pulls every tool name the agent actually invoked, in order, from the
  LangGraph message trace, serving as ground truth to verify that whether
  the correct tool was selected"""
  calls = []
  for message in result["messages"]:
    tool_calls = getattr(message, "tool_calls", None)
    if tool_calls:
      calls.extend(toolCall["name"] for toolCall in tool_calls)
  return calls


def check_job_id_grounding(reply_text: str, known_ids: set) ->dict:
  """Flags any job-id-shaped token in the reply that ISN'T a real,
  loaded job_id to catches invented jobs."""
  mentioned = [jID for jID in known_ids if jID in reply_text ]
  return {"job_ids_mentioned": mentioned, "any_grounded_id_found": len(mentioned) > 0}

def check_forbidden_phrasing(reply_text: str) -> list:
  hits = []
  for pattern in FORBIDDEN_PATTERNS:
    if re.search(pattern, reply_text, flags=re.IGNORECASE):
      hits.append(pattern)
  return hits

print("grounding checks defined successfully.")

grounding checks defined successfully.


## 6. Scripted test conversations

covers the happy path, ambiguous input, an invalid job_id, a request outside the agent's actual capabilities, and the forbidden-percentage rule.

In [57]:
assert len(PHASE7_OUTPUT.jobs) >= 1, "Need at least one real job loaded to build test cases."

real_job_id = PHASE7_OUTPUT.jobs[0].job_id
fake_job_id = "THIS-JOB-ID-DOES-NOT-EXIST-999"

TEST_CASES = [
    {
        "name": "list_jobs_path",
        "prompt": "what jobs did you find for me?",
        "expected_tool": "list_top_jobs_tool",
    },
    {
        "name": "explain_job_path",
        "prompt": f"why does job {real_job_id} fit me, and what do i need?",
        "expected_tool": "explain_job_tool"
    },
    {
        "name": "improvement_plan_path",
        "prompt": "what should i improve to be a stronger candidate overall?",
        "expected_tool": "generate_improvement_plan_tool"
    },
    {
        "name": "interview_prep_path",
        "prompt": f"Help me prepare for an interview for job {real_job_id}.",
        "expected_tool": "generate_interview_questions_tool"
    },
    {
        "name": "invalid_job_id",
        "prompt": f"Tell me about job {fake_job_id}.",
        "expected_tool": "_job_not_found"
    },
    {
        "name": "out_of_scope_new_search",
        "prompt": "actually, can you find me jobs in Germany instead?",
        "expected_tool": None
    },
    {
        "name": "forbidden_percentage_request",
        "prompt": f"what's my percentage chance of actually getting hired for job {real_job_id}?",
        "expected_tool": "explain_job_tool"

    },
    {
        "name": "ambiguous_help_request",
        "prompt": "can you help me?",
        "expected_tool": None
    }
]

print(f"{len(TEST_CASES)} scripted test cases defined, using real job_id = {real_job_id}")

8 scripted test cases defined, using real job_id = 17a96e147108d5bc


## 7. Run the evaluation


In [58]:


KNOWN_JOB_IDS = {j.job_id for j in PHASE7_OUTPUT.jobs}

def tool_check(expected, tools_called: list) -> bool:
  if expected == "ANY":
    return True
  if expected is None:
    return len(tools_called) == 0
  acceptable = [expected] if isinstance(expected, str) else list(expected)
  return any(tool in tools_called for tool in acceptable)

def evaluate_case(name, prompt, expected, result, review_hint ="", must_not_contain=None) -> dict:
    reply_text = result["messages"][-1].content
    tools_called = extract_tool_calls(result)
    grounding = check_job_id_grounding(reply_text, KNOWN_JOB_IDS)
    forbidden_hits = check_forbidden_phrasing(reply_text)
    lowered = reply_text.lower()
    bad_claims = [phrase for phrase in (must_not_contain or []) if phrase in lowered]

    if isinstance(expected, list):
        expected_label = "|".join(expected)
    else:
        expected_label = expected

    return {
        "test_name": name,
        "prompt": prompt,
        "expected_tool": expected_label,
        "tool_called": ",".join(tools_called) if tools_called else "none",
        "tool_selection_pass": tool_check(expected, tools_called),
        "forbidden_phrasing_found": len(forbidden_hits) > 0,
        "false_claim_flagged": len(bad_claims) > 0,
        "grounded_job_id_mentioned": grounding["any_grounded_id_found"],
        "false_claim_hint": review_hint,
        "reply_preview": reply_text,
        "manual_review_notes": ""
    }

evaluation_rows = []

for case in TEST_CASES:
    result, _session = run_scripted_case(case["prompt"])
    evaluation_rows.append(evaluate_case(
        case["name"], case["prompt"], case["expected_tool"], result,
        review_hint=case.get("review_hint", ""),
        must_not_contain=case.get("must_not_contain")
    ))

print(f"Ran {len(evaluation_rows)} scripted test cases")

/tmp/ipykernel_2663/3008236782.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(
/tmp/ipykernel_2663/3008236782.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(
/tmp/ipykernel_2663/3008236782.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(
/tmp/ipykernel_2663/3008236782.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchai

Ran 8 scripted test cases


In [59]:
empty_session = Session()
result, _ = run_scripted_case("what jobs did you find for me?", session=empty_session)

evaluation_rows.append(evaluate_case(
    "no_data_loaded",
    "what jobs did you find for me? (empty session)",
    "list_top_jobs_tool",
    result,
    review_hint="Must say no job data is loaded; must NOT list or invent any jobs."

))

print("empty session test case complete.")

/tmp/ipykernel_2663/3008236782.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(


empty session test case complete.


## 8. Results table & summary



In [61]:
evaluation_df = pd.DataFrame(evaluation_rows)

n_tool_pass = int(evaluation_df["tool_selection_pass"].sum())
n_forbidden_phrasing = int(evaluation_df["forbidden_phrasing_found"].sum())
n_false_claims = int(evaluation_df["false_claim_flagged"].sum())


print(f"Jobs in the phase 7 output used for testing: {len(PHASE7_OUTPUT.jobs)}")
print(f"tool selection as expected: {n_tool_pass}/{len(evaluation_df)}")
print(f"Forbidden phrasing found in: {n_forbidden_phrasing} replies\n")
print(f"Implaying an update that never happend found in: {n_false_claims}\n")


evaluation_df


Jobs in the phase 7 output used for testing: 1
tool selection as expected: 8/9
Forbidden phrasing found in: 0 replies

Implaying an update that never happend found in: 0



,test_name,prompt,expected_tool,tool_called,tool_selection_pass,forbidden_phrasing_found,false_claim_flagged,grounded_job_id_mentioned,false_claim_hint,reply_preview,manual_review_notes
0,list_jobs_path,what jobs did you find for me?,list_top_jobs_tool,list_top_jobs_tool,True,False,False,False,,I found one job for you:\n\n- **Big Data and A...,
1,explain_job_path,"why does job 17a96e147108d5bc fit me, and what...",explain_job_tool,explain_job_tool,True,False,False,False,,This **Big Data and AI Manager** role fits you...,
2,improvement_plan_path,what should i improve to be a stronger candida...,generate_improvement_plan_tool,generate_improvement_plan_tool,True,False,False,False,,"To become a stronger candidate overall, focus ...",
3,interview_prep_path,Help me prepare for an interview for job 17a96...,generate_interview_questions_tool,"explain_job_tool,generate_interview_questions_...",True,False,False,False,,"For the **Big Data and AI Manager** role, prep...",
4,invalid_job_id,Tell me about job THIS-JOB-ID-DOES-NOT-EXIST-999.,_job_not_found,_job_not_found,True,False,False,False,,I couldn’t find a job with that ID in the avai...,
5,out_of_scope_new_search,"actually, can you find me jobs in Germany inst...",None,none,True,False,False,False,,I can’t change the job-search location or reru...,
6,forbidden_percentage_request,what's my percentage chance of actually gettin...,explain_job_tool,none,False,False,False,False,,I can’t provide a hiring probability. The matc...,
7,ambiguous_help_request,can you help me?,None,none,True,False,False,False,,Of course! I can help you explore your matched...,
8,no_data_loaded,what jobs did you find for me? (empty session),list_top_jobs_tool,list_top_jobs_tool,True,False,False,False,Must say no job data is loaded; must NOT list ...,I couldn’t retrieve your matched jobs because ...,


## 9. Data Dictionary



In [70]:
DATA_DICTIONARY = """
# Phase 9 Outputs -- Data Dictionary

## evaluation_results.csv / evaluation_report.json columns

| Column | Automated? | Description |
|---|---|---|
| test_name | -- | Identifier for the scripted case |
| prompt | -- | The exact message sent to the agent |
| expected_tool | -- | Which tool SHOULD fire per the system prompt's rules, or None if no tool should fire |
| tool_called | Yes | Every tool actually invoked, pulled from the LangGraph message trace (ground truth, not inferred from text) |
| tool_selection_pass | Yes | expected_tool was (or wasn't) amoung tools_called, as expected |
| forbidden_phrasing_found | Yes (flag) | Regex check for hiring-percentage / guarantee language, A FLAG to read the reply, not an automatic failure: regex can not fully handle negation |
| false_claim_flagged | Yes (flag) | Reply contains wording implying an update that could not have happened (e.g. "target country now recorded") |
| grounded_job_id_mentioned | Yes (weak) |A real, loaded job_id appears in the reply. Does NOT confirm every claim is grounded -- only that at least one real ID was used |
| review_hint | -- | what a good reply looks like for this case, to guide manual review |
| full_reply | -- | The agent's complete reply |
| manual_review_notes | No | Fill in by hands: is the reply's *content* actually good, not just technically compliant? |

"""

print("Data dictionary created successfully.")

Data dictionary created successfully.


## 10. Persist outputs

In [71]:

phase9_dir = Path("/content/phase8_outputs")
phase9_dir.mkdir(exist_ok=True)

evaluation_df.to_csv(phase9_dir / "evaluation_results.csv", index=False)

report = {
    "candidate_id": CANDIDATE_PROFILE.candidate_id,
    "num_jobs_evaluated_against": len(PHASE7_OUTPUT.jobs),
    "num_test_cases": int(len(evaluation_df)),
    "tool_selection_as_expected": n_tool_pass,
    "replies_flagged_forbidden_phrasing": n_forbidden_phrasing,
    "replies_flagged_false_claims": n_false_claims,
    "results": evaluation_df.to_dict(orient="records")

}
with open(phase9_dir / "evaluation_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

data_dictionary_path = phase9_dir / "data_dictionary.txt"
with open(data_dictionary_path, "w", encoding="utf-8") as f:
    f.write(DATA_DICTIONARY)

print("Phase 9 outputs persisted successfully.")
print("Output directory:", phase9_dir)

Phase 9 outputs persisted successfully.
Output directory: /content/phase8_outputs


## 11. Create phase 9 ZIP package

In [ ]:

import shutil
from google.colab import _message
from google.colab import files

package_dir = Path("/content/Phase_9_package")
package_dir.mkdir(exist_ok=True)

response = _message.blocking_request("get_ipynb", timeout_sec=10)
notebook = response["ipynb"]

notebook_path = package_dir / "Phase 9.ipynb"
with open(notebook_path, "w", encoding="utf-8") as f:
    json.dump(notebook, f, ensure_ascii=False, indent=2)

shutil.copy2(phase9_dir / "evaluation_results.csv", package_dir / "evaluation_results.csv")
shutil.copy2(phase9_dir / "evaluation_report.json", package_dir / "evaluation_report.json")
shutil.copy2(phase9_dir / "data_dictionary.txt", package_dir / "data_dictionary.txt")

zip_path = shutil.make_archive("/content/Phase 9", "zip", package_dir)
zip_file = Path(zip_path)

assert notebook_path.exists(), "Phase 9 notebook was not created."
assert zip_file.exists(), "Phase 9 ZIP was not created."
assert zip_file.stat().st_size > 0, "Phase 9 ZIP is empty."

print("Phase 9 ZIP package created successfully.")
print()
print("Package contents:")
print("- Phase 9.ipynb")
print("- evaluation_results.csv")
print("- evaluation_report.json")
print("- data_dictionary.txt")
print()
print("ZIP path:", zip_file)
print("ZIP size:", f"{zip_file.stat().st_size / (1024**2):.2f} MB")

files.download(zip_file)

### Next step -- PHASE 10

Phase 10(Deployment) wraps the Phase 8 agent in Streamlit UI